# Lesson 16: Image Finder and AIO Analyzer

The last lesson built the **Content Writer** — the main agent that researches and writes articles. Now the 2 supporting agents:

- **Image Finder** — Finds relevant images and inserts them into articles
- **AIO Analyzer** — Analyzes Google AI Overviews and suggests content optimizations

Both use Claude Sonnet, same as the Content Writer. Together with the Content Writer, these 3 agents form the complete team.

## First: What is Markdown?

Articles are stored in **Markdown** — a text format for creating formatted documents. All files in the `content/` folder are Markdown (`.md`) files.

| What you type | What it looks like |
|---|---|
| `# Title` | Big heading (H1) |
| `## Section` | Medium heading (H2) |
| `### Subsection` | Small heading (H3) |
| `**bold text**` | **bold text** |
| `- item` | Bullet point |
| `1. item` | Numbered list |
| `[link text](url)` | Clickable link |
| `![alt text](image-url)` | An image |

**Why Markdown?**
- Plain text — easy to store, version, and process in code
- Converts to HTML for websites with one command
- Standard format for content management systems
- This notebook's text cells are Markdown

The Image Finder's job is to insert `![alt text](image-url)` lines into existing Markdown articles.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output/backend"))

from dotenv import load_dotenv
load_dotenv()

## Image Finder — Production Code

The Image Finder reads an existing article, searches for relevant images, and inserts them as Markdown image lines.

Its workflow:
1. `get_article_content(article_id)` — read the article
2. Search for images matching each section's topic (via DataForSEO Images API)
3. Insert `![alt text](url)` after relevant `##` headings
4. `update_article_content(article_id, updated_markdown)` — save the updated article

Here's the actual code from `output/backend/agents/image_finder.py`:

In [ ]:
# Show the actual production code
image_finder_path = os.path.abspath("../../output/backend/agents/image_finder.py")
with open(image_finder_path, "r", encoding="utf-8") as f:
    print(f.read())

## Image Finder — Key Patterns

Notice two patterns in the code above:

### 1. Factory function: `build_image_finder()`

The Image Finder is wrapped in a function that returns `None` if DataForSEO isn't configured:

```python
def build_image_finder() -> Agent | None:
    creds = get_dataforseo_credentials()
    if not creds:
        return None
    return Agent(...)
```

This is called a **factory function** — it builds and returns an object. The `-> Agent | None` means it returns either an Agent or None.

### 2. Read-modify-write pattern

The Image Finder doesn't write articles from scratch. It:
1. **Reads** an existing article (`get_article_content`)
2. **Modifies** it by inserting image lines
3. **Writes** the updated version back (`update_article_content`)

This is a common pattern: one agent creates, another agent enhances.

In [ ]:
# Build the Image Finder (same as production code)
from agents.image_finder import image_finder

if image_finder:
    print(f"Image Finder: ACTIVE")
    print(f"  Model: Claude Sonnet")
    print(f"  Tools: {len(image_finder.tools)}")
    for t in image_finder.tools:
        name = getattr(t, 'name', None) or getattr(t, '__name__', str(t))
        print(f"    - {name}")
else:
    print("Image Finder: INACTIVE (no DataForSEO key)")
    print("This is fine — the team will skip image insertion.")

## Optional by Design

The Image Finder is **intentionally optional**. When the team is assembled in `team.py`:

```python
members = [content_writer, aio_analyzer]
if image_finder is not None:
    members.append(image_finder)
```

No DataForSEO key? `build_image_finder()` returns `None`, and the team simply has 2 members instead of 3. No errors, no special handling needed.

This is a good design principle: **optional features should degrade gracefully**. The product works without image API keys — articles are created normally, just without images.

## AIO Analyzer — Production Code

**AIO** stands for **AI Overview** — the AI-generated summary boxes that Google shows at the top of some search results. For SEO teams, understanding and optimizing for these overviews is critical.

The AIO Analyzer has 2 tools:
- `analyze_keyword_aio(keyword)` — Check what Google's AI says about a topic
- `optimize_for_aio(article_id, keyword)` — Compare an article against current AI Overviews and suggest improvements

Here's the actual code from `output/backend/agents/aio_analyzer.py`:

In [ ]:
# Show the actual production code
aio_analyzer_path = os.path.abspath("../../output/backend/agents/aio_analyzer.py")
with open(aio_analyzer_path, "r", encoding="utf-8") as f:
    print(f.read())

## AIO Analyzer — Key Pattern

The AIO Analyzer is simpler than the other agents — it's always active (no conditional setup) and uses plain functions as tools:

```python
tools=[analyze_keyword_aio, optimize_for_aio]
```

These aren't toolkit classes like `DataForSEOSearchTools` or `DataForSEOImageTools`. They're regular Python functions passed directly as tools. Agno supports both patterns:
- **Toolkit classes** — for groups of related tools (search, images)
- **Plain functions** — for individual tool functions (analyze, optimize)

The AIO Analyzer's instructions enforce a two-part response format: first the raw AIO data (verbatim), then analysis and suggestions. This ensures the user always sees the original data alongside the agent's interpretation.

In [ ]:
from agents.aio_analyzer import aio_analyzer

print(f"AIO Analyzer: ACTIVE")
print(f"  Model: Claude Sonnet")
print(f"  Tools: {len(aio_analyzer.tools)}")
for t in aio_analyzer.tools:
    name = getattr(t, '__name__', str(t))
    print(f"    - {name}")

## All 3 Agents — Summary

Our product has **3 specialized agents**, all using Claude Sonnet:

| Agent | Role | Tools | Optional? |
|-------|------|-------|-----------|
| **Content Writer** | Research topics, write SEO articles | DataForSEO search + storage | No (core agent) |
| **Image Finder** | Find and insert images into articles | DataForSEO images + storage | Yes (needs DataForSEO key) |
| **AIO Analyzer** | Analyze Google AI Overviews | AIO analysis functions | No (but AIO tools need DataForSEO) |

```
Your request --> [Team Leader] --> delegates to the right agent
                                    |
                                    +-- Content Writer  (research + write + save)
                                    +-- Image Finder    (read + search images + update)
                                    +-- AIO Analyzer    (analyze AIO + suggest optimizations)
```

**Key takeaways:**
- All agents use **Claude Sonnet** — one model, one API key
- The Image Finder is **optional** (graceful degradation)
- Each agent has **focused tools** for its specific job
- Agents follow a pattern: model + tools + instructions

**Next lesson:** Connect all 3 agents into a coordinated team with batch processing.

## Exercise

Quality check on Markdown articles. Given an article stored in `content/`, write code to:

1. Read the article using `get_article_content(article_id)`
2. Count how many images (`![`) are in the article
3. Count how many H2 headings (`## `) are in the article
4. Check if every image has alt text (not `![](url)` — the brackets should contain text)

**Helpful methods:**
- `.count(substring)` counts how many times a substring appears in a string. Example: `"hello world hello".count("hello")` returns `2`.
- `"![](url)" in text` checks if a substring exists in the string.

In [ ]:
# Exercise: Quality check on a Markdown article
import json
from tools.storage import get_article_content, list_all_articles

# First, check if we have any articles
result = list_all_articles()
articles = json.loads(result)
if not articles:
    print("No articles found. Run Lesson 15 first to create one.")
else:
    article_id = articles[-1]["id"]
    print(f"Checking article: {article_id}\n")

    content_json = get_article_content(article_id)
    content = json.loads(content_json)
    article = content["article_markdown"]

    # 1. Count images
    image_count = article.___(___) 
    print(f"Images: {image_count}")

    # 2. Count H2 headings
    h2_count = article.___(___)
    print(f"H2 headings: {h2_count}")

    # 3. Check for images without alt text
    has_empty_alt = "![](___)" in article  # Fix this pattern
    print(f"Has images without alt text: {has_empty_alt}")

<details>
<summary>Click to reveal answer</summary>

```python
import json
from tools.storage import get_article_content, list_all_articles

result = list_all_articles()
articles = json.loads(result)
article_id = articles[-1]["id"]
print(f"Checking article: {article_id}\n")

content_json = get_article_content(article_id)
content = json.loads(content_json)
article = content["article_markdown"]

# 1. Count images
image_count = article.count("![")
print(f"Images: {image_count}")

# 2. Count H2 headings
h2_count = article.count("## ")
print(f"H2 headings: {h2_count}")

# 3. Check for images without alt text
has_empty_alt = "![]()" in article or "![ ](" in article
print(f"Has images without alt text: {has_empty_alt}")
```
</details>